# Week 3: Handling Missing Values and Data Types - The Penguins Dataset

## Python Concepts Introduced
- Loops (for loops, but shown as inferior to Pandas methods).
- Pandas methods: .fillna(), .dropna(), .astype(), .groupby(), .agg().
- Categorical data type in Pandas.
- More Seaborn plotting: sns.boxplot().
- Basic functions (simple def for reusable code).

## Dataset Context
The Penguins dataset includes measurements from 344 penguins of three species: Adelie, Chinstrap, and Gentoo, observed on islands in Antarctica. 
The target variable could be the species (for classification), or body_mass_g (for regression later).
The features include:
- species: Adelie, Chinstrap, Gentoo (string)
- island: Biscoe, Dream, Torgersen (string)
- bill_length_mm: Length of the bill (float, some missing)
- bill_depth_mm: Depth of the bill (float, some missing)
- flipper_length_mm: Length of the flipper (float, some missing)
- body_mass_g: Body mass in grams (float, some missing)
- sex: male or female (string, some missing)
- year: Year of observation (integer)
This data comes from real ecological research and helps explore how physical traits differ by species or island. Imagine it as studying wildlife: cleaning the data ensures accurate comparisons, like average flipper length per species.

### Step 1: Loading the Dataset - Using Seaborn for Built-in Data
We load from Seaborn because it has convenient datasets with some natural missing values. This builds on previous loading but introduces a new source.

In [ ]:
# Import necessary libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the Penguins dataset
penguins_df = sns.load_dataset('penguins')

# Display the first 5 rows
display(penguins_df.head())
print("\n")
print(f"penguins_df shape : {penguins_df.shape}")

# Output interpretation
Table with columns like species, island, bill_length_mm, etc. Some rows may show NaN in sex or measurements. This confirms loading and shows real missings (e.g., about 11 missings in numerical features, more in sex).

### Step 2: Reviewing Missing Values - Building on Inspection
We re-check missings to set the stage for handling. This reinforces last week's skills and why we clean now: missings can break calculations or models.

In [ ]:
# Count missing values per column
missing_values = penguins_df.isnull().sum()
print(missing_values)

# Percentage of missing
missing_pct = (missing_values / len(penguins_df)) * 100
print(missing_pct)

In [ ]:
penguins_df[penguins_df.isnull().any(axis=1)]

# Output interpretation
bill_length_mm: 2, sex: 11, etc. Percentages low (~0.58% for bills, ~3.2% for sex). This suggests we can fill or drop without losing much data, but dropping all would remove rows with any missing.

### Step 3: Handling Missing Values - Dropping Rows
Dropping is simple for small missings. We do this first to show when it's appropriate (low % missings), before more advanced filling.

In [ ]:
# Drop rows with any missing values
penguins_clean_drop = penguins_df.dropna()

# Check shape before and after
print('Original shape:', penguins_df.shape)
print('After drop:', penguins_clean_drop.shape)

# Check missings now
print(penguins_clean_drop.isnull().sum())

# Output interpretation
Original (344, 7), after ~ (333, 7) – lost 11 rows. No missings left. Notice we preserved most data, but if missings were correlated (e.g., all from one species), this could bias.

### Step 4: Handling Missing Values - Filling with Means/Modes
Filling preserves rows. Use mean for numbers (average), mode for categories. This introduces why: to avoid data loss in small datasets.

In [ ]:
penguins_df.describe()

In [ ]:
mode_sex = penguins_df['sex'].mode()[0]
mode_island = penguins_df['island'].mode()[0]
print(mode_sex)
print(mode_island)
print("\n")

print(penguins_df["sex"].value_counts())
print(penguins_df["island"].value_counts())

In [ ]:
print(f"Before: {penguins_df.isnull().sum()}")

# Fill numerical missings with column means
numerical_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
penguins_df[numerical_cols] = penguins_df[numerical_cols].fillna(penguins_df[numerical_cols].mean())

# Fill categorical 'sex' with mode (most common)
mode_sex = penguins_df['sex'].mode()[0]
penguins_df['sex'] = penguins_df['sex'].fillna(mode_sex)

# # Check missings
print(f"After: {penguins_df.isnull().sum()}")
print("Shape after imputation:", penguins_df.shape)

# Output interpretation
All missings now 0. Means used for measurements (e.g., avg bill length ~43.9), mode 'male' for sex (since more males). This keeps all 344 rows but imputes values – good for ML but note potential bias if missings aren't random.

### Step 5: Converting Data Types - To Categories
Convert strings to 'category' for efficiency and proper handling in ML. Introduce now because after cleaning, types matter for grouping/plots.

In [ ]:
# Check current types
print(penguins_df.dtypes)

# Convert species, island, sex to category
categorical_cols = ['species', 'island', 'sex']
penguins_df[categorical_cols] = penguins_df[categorical_cols].astype('category')

# Check again
print(penguins_df.dtypes)

# Output interpretation
Originally object for strings, now category. This saves memory and tells Pandas these are finite options (e.g., only 3 species), useful for encoding later.

### Step 6: Introducing Loops - But Why Avoid Them
Show a loop for filling missings manually, then contrast with vectorized .fillna(). Teach loops contextually but emphasize Pandas efficiency.

In [ ]:
# Reload to have missings again for demo
penguins_df = sns.load_dataset('penguins')

# Slow loop way (for illustration)
for col in numerical_cols:
    mean_val = penguins_df[col].mean()
    for i in range(len(penguins_df)):
        if pd.isnull(penguins_df.loc[i, col]):
            penguins_df.loc[i, col] = mean_val

# Now vectorized (faster)
penguins_df[numerical_cols] = penguins_df[numerical_cols].fillna(penguins_df[numerical_cols].mean())

# Note: The loop does the same but is slower for large data.

# Output interpretation
No direct output, but understand: Loop iterates row-by-row (inefficient), vectorized applies to whole column at once. For 344 rows, similar; for millions, loop slow. Always prefer Pandas methods.

### Step 7: Grouping and Aggregation - Summarizing Subgroups
Group by species to get means, etc. This comes after cleaning because aggregates ignore missings, but we filled for completeness.

In [ ]:
# Group by species and aggregate means
grouped = penguins_df.groupby('sex').agg({
    'bill_length_mm': 'mean',
    'body_mass_g': 'mean'
})
print(grouped)

# More aggregates
grouped_multi = penguins_df.groupby('species').agg({
    'bill_length_mm': ['mean', 'std'],
    'body_mass_g': 'count'
})
print(grouped_multi)

# Output interpretation
Table: Adelie avg bill ~38.8, mass ~3700; Gentoo longer bill ~47.5, heavier ~5076. Std shows spread. Count confirms group sizes. This reveals species differences intuitively.

### Step 8: Visualizing Groups - Box Plots
Box plots show distributions per group. Builds on visuals, now with groups.

In [ ]:
# Box plot for body mass by species
plt.figure(figsize=(8, 5))
sns.boxplot(x='species', y='body_mass_g', data=penguins_df)
plt.title('Body Mass by Penguin Species')
plt.show()

# Output interpretation
Boxes: Gentoo highest median mass, wider range; Adelie lowest. Outliers as dots. This visually confirms aggregates: species vary in size.

### Step 9: Simple Functions - Reusable Cleaning
Introduce functions to wrap cleaning steps, for reproducibility.

In [ ]:
# Define a function to clean data
def clean_penguins(df):
    df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].mean())
    df['sex'] = df['sex'].fillna(df['sex'].mode()[0])
    df[categorical_cols] = df[categorical_cols].astype('category')
    return df

# Use it
penguins_df = sns.load_dataset('penguins')
cleaned_df = clean_penguins(penguins_df)
print(cleaned_df.isnull().sum())

In [ ]:
penguins_df.values
print(type(penguins_df.values))

# Output interpretation
All 0 missings. Function makes code reusable: call on any similar DF. This promotes professional habits.

## ⚠️ Common Mistakes Box
- Filling before checking: Impute wrong if types incorrect or outliers present.
- Using mean for skewed data: Better median sometimes, but we keep simple.
- Loops on DataFrames: Slow; always vectorize with Pandas.
- Grouping with missings: Aggregates skip NaN, but filling ensures consistency.
- Wrong astype: Converting numbers to category loses math ability.
- Forgetting return in functions: Function does nothing without it.

## Guided Exercises
Fill in TODOs. Run and check.

In [ ]:
# Exercise 1: Drop missings in 'sex' only

penguins_no_sex_missing = penguins_df.None(subset=None)
print(penguins_no_sex_missing.shape)

# Expected: (333, 7) or similar

In [ ]:
# Exercise 2: Group by island, mean flipper_length_mm
# TODO: groupby('island').agg({'flipper_length_mm': 'mean'})
island_group = penguins_df.None('island').agg({None: None})
print(island_group)

# Expected: Table with means ~ Biscoe 210, Dream 190, Torgersen 191

In [ ]:
# Exercise 3: Function to print summary
# TODO: def print_summary(df): print(df.describe())
None print_summary(df):
    None(df.describe())

# Call it
print_summary(penguins_df)

# Expected: .describe() output

## Reflection Questions
- Why fill missings instead of dropping? (Hint: Data loss in small sets.)
- What happens if you loop on a large DF? Try timing if curious.
- Why group after cleaning? Imagine aggregating with missings.
- How does categorizing help ML? (Hint: Encoding, memory.)